# R5 - Verificación de las ROI materializadas (condiciones B y C)

Este notebook verifica, **sin escribir ni regenerar ningún archivo**, que las 2000 ROI ya materializadas para `train` y `validation` corresponden a la lógica actual documentada de extracción de ROI:

- **B (ROI predicha):** máscara predicha por el U-Net baseline congelado; se conserva únicamente la componente conexa mayor; fallback a imagen completa si la máscara está vacía o la componente mayor ocupa menos del 0.1 % de la imagen; bounding box con margen del 10 %.
- **C (ROI de referencia):** máscara de referencia; se conservan todas las componentes; sin fallback; bounding box con margen del 10 %.

Para cada ROI se recalcula la geometría esperada a partir de la **máscara persistida** y de la imagen original, y se compara con el manifiesto y con el PNG existente. El conjunto `test` no se utiliza.

Si alguna comprobación falla, la verificación se detiene e indica la condición, la partición y el `image_id` afectados. No se corrige ni se regenera nada.

In [1]:
from pathlib import Path
import sys

import cv2
import numpy as np
import pandas as pd
from PIL import Image


def locate_project_root() -> Path:
    current_dir = Path.cwd().resolve()

    for candidate in [current_dir, *current_dir.parents]:
        if (
            (candidate / "src").exists()
            and (candidate / "data").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se pudo localizar la raíz del proyecto."
    )


PROJECT_ROOT = locate_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.segmentation.roi import (
    MIN_COMPONENT_AREA_FRACTION,
    expand_bbox,
    extract_roi_from_mask,
    keep_largest_component,
    mask_to_bbox,
    resize_mask_to_image,
)

## 1. Configuración

Los valores esperados se fijan a partir de las decisiones documentadas de R4 (análisis de ROI sobre `validation`) y de la auditoría de R5.

In [2]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

EXPERIMENTAL_DIR = DATA_DIR / "MMOTU_OTU_2D_experimental"
ROI_DIRS = {
    "B": DATA_DIR / "MMOTU_OTU_2D_roi_pred",
    "C": DATA_DIR / "MMOTU_OTU_2D_roi_gt",
}

PRED_MASKS_DIR = PROJECT_ROOT / "results" / "segmentation" / "unet_baseline"
MANIFEST_DIR = PROJECT_ROOT / "results" / "classification" / "roi_generation"
MANIFEST_PATHS = {
    "B": MANIFEST_DIR / "manifest_B_roi_pred.csv",
    "C": MANIFEST_DIR / "manifest_C_roi_gt.csv",
}

SPLIT_CSV = PROJECT_ROOT / "configs" / "splits" / "mmotu_experimental_split_standard.csv"

CONDITIONS = ["B", "C"]
SPLITS = ["train", "validation"]  # test no se utiliza en R5
EXPECTED_COUNTS = {"train": 800, "validation": 200}

MARGIN_RATIO = 0.10
MIN_AREA_FRACTION = 0.001
PRED_MASK_SIZE = (256, 256)  # (alto, ancho) de las máscaras predichas persistidas

# Fallbacks esperados en B
EXPECTED_B_FALLBACK_IDS = {"train": set(), "validation": {775, 869}}
EXPECTED_B_FALLBACK_REASON = "tiny_component"

assert "test" not in SPLITS
assert MIN_COMPONENT_AREA_FRACTION == MIN_AREA_FRACTION, "MIN_COMPONENT_AREA_FRACTION de roi.py cambió"

MANIFEST_COLUMNS = [
    "image_id", "split", "condition", "source_image", "source_mask",
    "roi_width", "roi_height", "roi_area_fraction", "used_fallback", "fallback_reason", "output_path",
]

print("Raíz del proyecto:", PROJECT_ROOT)

Raíz del proyecto: /home/jvillanueva/projects/ovarian-tumor-ultrasound-classification


## 2. Manifiestos y particiones

Se comprueba que cada manifiesto contiene exactamente las muestras de `train` (800) y `validation` (200) del manifiesto de particiones, sin duplicados ni particiones adicionales.

In [3]:
split_df = pd.read_csv(SPLIT_CSV)

csv_by_split = {
    split: split_df[split_df["experimental_split"] == split].set_index("image_id")
    for split in SPLITS
}

for split in SPLITS:
    assert len(csv_by_split[split]) == EXPECTED_COUNTS[split], split

manifests = {}

for condition in CONDITIONS:
    manifest = pd.read_csv(MANIFEST_PATHS[condition], dtype={"image_id": int})

    assert list(manifest.columns) == MANIFEST_COLUMNS, (condition, list(manifest.columns))
    manifest["fallback_reason"] = manifest["fallback_reason"].fillna("")

    assert set(manifest["split"]) == set(SPLITS), (condition, set(manifest["split"]))
    assert (manifest["condition"] == condition).all(), condition
    assert not manifest.duplicated(["split", "image_id"]).any(), condition

    for split in SPLITS:
        manifest_ids = set(manifest.loc[manifest["split"] == split, "image_id"])
        csv_ids = set(csv_by_split[split].index)
        assert manifest_ids == csv_ids, f"{condition} {split}: image_id del manifiesto distintos de los del CSV de particiones"
        assert len(manifest_ids) == EXPECTED_COUNTS[split]

    manifests[condition] = manifest

print("Manifiestos: image_id y particiones consistentes con el CSV de particiones")

Manifiestos: image_id y particiones consistentes con el CSV de particiones


## 3. Función de verificación por muestra

Para cada ROI se comprueba, en este orden:

1. **Trazabilidad:** `split` y `condition` del manifiesto; existencia de la imagen original, de la máscara utilizada y de la ROI; rutas del manifiesto coincidentes con los archivos esperados.
2. **Máscara y dimensiones originales:** valores de la máscara; tamaño de la máscara de referencia igual al de la imagen (C) o 256×256 (B).
3. **Geometría esperada:** la máscara persistida se lleva a la resolución original; en B se conserva la componente mayor y se evalúa el fallback; se calcula el bounding box y el margen del 10 %.
4. **Dimensiones:** esperadas = registradas en el manifiesto = reales del PNG.
5. **Fallback:** coincidencia con el manifiesto y con la lógica actual (`extract_roi_from_mask`).
6. **Contenido:** el PNG es idéntico al recorte esperado de la imagen original.

In [4]:
class VerificationError(Exception):
    pass


def fail(condition, split, image_id, message):
    raise VerificationError(f"[Condición {condition} | {split} | image_id={image_id}] {message}")


def count_components(mask):
    n_labels, _ = cv2.connectedComponents(mask.astype(np.uint8), connectivity=8)
    return n_labels - 1


def verify_sample(condition, split, row, image_name):
    """Verifica una ROI. Lanza VerificationError ante la primera discrepancia."""
    image_id = int(row["image_id"])
    stem = Path(image_name).stem

    # 1. Trazabilidad
    if int(stem) != image_id:
        fail(condition, split, image_id, f"image_name del CSV ({image_name}) no corresponde al image_id")
    if row["split"] != split:
        fail(condition, split, image_id, f"split del manifiesto = {row['split']}")
    if row["condition"] != condition:
        fail(condition, split, image_id, f"condition del manifiesto = {row['condition']}")

    image_path = EXPERIMENTAL_DIR / split / "images" / image_name
    roi_path = ROI_DIRS[condition] / split / "images" / f"{stem}.png"
    if condition == "B":
        mask_path = PRED_MASKS_DIR / split / "predicted_masks" / f"{stem}_pred.png"
    else:
        mask_path = EXPERIMENTAL_DIR / split / "masks" / f"{stem}_binary.PNG"

    for label, path in (("imagen original", image_path), ("máscara utilizada", mask_path), ("ROI", roi_path)):
        if not path.is_file():
            fail(condition, split, image_id, f"no existe la {label}: {path}")

    for column, path in (("source_image", image_path), ("source_mask", mask_path), ("output_path", roi_path)):
        if Path(row[column]).parts[-4:] != path.parts[-4:]:
            fail(condition, split, image_id, f"{column} del manifiesto ({row[column]}) no corresponde a {path}")

    # 2. Imagen original y máscara persistida
    with Image.open(image_path) as original:
        image = original.convert("RGB")
    width, height = image.size

    with Image.open(mask_path) as persisted:
        mask_array = np.asarray(persisted.convert("L"), dtype=np.uint8)

    allowed_values = {0, 255} if condition == "B" else {0, 1}
    if not set(np.unique(mask_array).tolist()) <= allowed_values:
        fail(condition, split, image_id, f"valores inesperados en la máscara: {np.unique(mask_array).tolist()}")

    if condition == "B":
        if mask_array.shape != PRED_MASK_SIZE:
            fail(condition, split, image_id, f"la máscara predicha mide {mask_array.shape}, se esperaba {PRED_MASK_SIZE}")
    else:
        if mask_array.shape != (height, width):
            fail(condition, split, image_id, f"la máscara de referencia mide {mask_array.shape} y la imagen {(height, width)}")

    mask_binary = (mask_array > 0).astype(np.uint8)

    # 3. Geometría esperada a partir de la máscara persistida
    resized = resize_mask_to_image(mask_binary, image_size=(width, height))
    n_components_before = count_components(resized)

    if condition == "B":
        mask_used = keep_largest_component(resized)

        # keep_largest=True: una sola componente, y es la de mayor área
        if resized.sum() > 0:
            _, _, stats, _ = cv2.connectedComponentsWithStats(resized, connectivity=8)
            largest_area = int(stats[1:, cv2.CC_STAT_AREA].max())
            if count_components(mask_used) != 1 or int(mask_used.sum()) != largest_area:
                fail(condition, split, image_id, "la máscara utilizada no es la componente conexa mayor")
        elif mask_used.sum() != 0:
            fail(condition, split, image_id, "máscara vacía con contenido tras keep_largest")

        if mask_used.sum() == 0:
            expected_fallback, expected_reason = True, "empty_mask"
        elif mask_used.sum() / (width * height) < MIN_AREA_FRACTION:
            expected_fallback, expected_reason = True, "tiny_component"
        else:
            expected_fallback, expected_reason = False, ""
    else:
        mask_used = resized  # se conservan todas las componentes
        if mask_used.sum() == 0:
            fail(condition, split, image_id, "la máscara de referencia está vacía")
        expected_fallback, expected_reason = False, ""

    if expected_fallback:
        expected_bbox = (0, 0, width, height)
    else:
        ys, xs = np.nonzero(mask_used)
        tight = (int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1)
        if tight != mask_to_bbox(mask_used):
            fail(condition, split, image_id, f"bounding box mínima distinta: {tight} vs {mask_to_bbox(mask_used)}")

        # Margen del 10 % del tamaño de la caja por lado, recortado a los bordes de la imagen
        margin_x = int(round((tight[2] - tight[0]) * MARGIN_RATIO))
        margin_y = int(round((tight[3] - tight[1]) * MARGIN_RATIO))
        expected_bbox = (
            max(0, tight[0] - margin_x),
            max(0, tight[1] - margin_y),
            min(width, tight[2] + margin_x),
            min(height, tight[3] + margin_y),
        )
        if expected_bbox != expand_bbox(tight, (width, height), MARGIN_RATIO):
            fail(condition, split, image_id, "margen del 10 % distinto del de roi.expand_bbox")

    expected_width = expected_bbox[2] - expected_bbox[0]
    expected_height = expected_bbox[3] - expected_bbox[1]

    # 4. Dimensiones: esperadas = manifiesto = PNG real
    if (int(row["roi_width"]), int(row["roi_height"])) != (expected_width, expected_height):
        fail(
            condition, split, image_id,
            f"dimensiones del manifiesto {(row['roi_width'], row['roi_height'])} != esperadas {(expected_width, expected_height)}",
        )

    with Image.open(roi_path) as roi_image:
        roi_size = roi_image.size
        roi_array = np.asarray(roi_image.convert("RGB"))

    if roi_size != (expected_width, expected_height):
        fail(condition, split, image_id, f"dimensiones del PNG {roi_size} != esperadas {(expected_width, expected_height)}")

    expected_fraction = (expected_width * expected_height) / (width * height)
    if abs(float(row["roi_area_fraction"]) - expected_fraction) > 1e-6:
        fail(condition, split, image_id, f"roi_area_fraction del manifiesto {row['roi_area_fraction']} != esperada {expected_fraction:.6f}")

    # 5. Fallback: manifiesto y lógica actual de roi.py
    if bool(row["used_fallback"]) != expected_fallback or row["fallback_reason"] != expected_reason:
        fail(
            condition, split, image_id,
            f"fallback del manifiesto ({row['used_fallback']}, '{row['fallback_reason']}') "
            f"!= esperado ({expected_fallback}, '{expected_reason}')",
        )

    if condition == "B":
        result = extract_roi_from_mask(
            image=image, binary_mask=mask_binary, margin_ratio=MARGIN_RATIO,
            keep_largest=True, fallback_to_full_image=True,
            min_component_area_fraction=MIN_COMPONENT_AREA_FRACTION,
        )
    else:
        result = extract_roi_from_mask(
            image=image, binary_mask=mask_binary, margin_ratio=MARGIN_RATIO,
            keep_largest=False, fallback_to_full_image=False,
            min_component_area_fraction=0.0,
        )
    if result.bbox != expected_bbox or result.used_fallback != expected_fallback:
        fail(condition, split, image_id, f"extract_roi_from_mask devuelve {result.bbox} (fallback={result.used_fallback}), esperado {expected_bbox}")

    # 6. Contenido exacto del recorte
    if not np.array_equal(np.asarray(image.crop(expected_bbox)), roi_array):
        fail(condition, split, image_id, "el contenido del PNG no coincide con el recorte esperado de la imagen original")

    return {
        "fallback": expected_fallback,
        "reason": expected_reason,
        "n_components_before": n_components_before,
    }

## 4. Verificación de las 2000 ROI

Se recorre cada combinación condición × partición. La verificación se detiene en la primera discrepancia.

In [5]:
stats = {}
fallback_ids = {}

for condition in CONDITIONS:
    for split in SPLITS:
        rows = manifests[condition][manifests[condition]["split"] == split].sort_values("image_id")
        n_verified = 0
        n_multicomponent = 0
        fallback_ids[(condition, split)] = []

        for row in rows.to_dict("records"):
            image_id = int(row["image_id"])
            image_name = csv_by_split[split].loc[image_id, "image_name"]

            details = verify_sample(condition, split, row, image_name)

            if condition == "B" and details["fallback"]:
                if image_id not in EXPECTED_B_FALLBACK_IDS[split]:
                    fail(condition, split, image_id, "fallback inesperado en B")
                if details["reason"] != EXPECTED_B_FALLBACK_REASON:
                    fail(condition, split, image_id, f"motivo de fallback inesperado: {details['reason']}")
            if condition == "C" and details["fallback"]:
                fail(condition, split, image_id, "fallback en C")

            if details["fallback"]:
                fallback_ids[(condition, split)].append(image_id)
            if details["n_components_before"] > 1:
                n_multicomponent += 1
            n_verified += 1

        stats[(condition, split)] = {
            "verified": n_verified,
            "expected": EXPECTED_COUNTS[split],
            "multicomponent": n_multicomponent,
        }

# Los fallbacks de B deben ser exactamente los esperados (ni más ni menos)
for split in SPLITS:
    observed = set(fallback_ids[("B", split)])
    if observed != EXPECTED_B_FALLBACK_IDS[split]:
        raise VerificationError(
            f"[Condición B | {split}] fallbacks observados {sorted(observed)} != esperados {sorted(EXPECTED_B_FALLBACK_IDS[split])}"
        )

## 5. Resumen de la verificación

In [6]:
total_verified = sum(item["verified"] for item in stats.values())
total_expected = sum(item["expected"] for item in stats.values())

assert total_verified == total_expected == 2000

print("Verificación de ROI (solo lectura)")
print("-" * 60)
for condition in CONDITIONS:
    for split in SPLITS:
        item = stats[(condition, split)]
        print(f"muestras verificadas {condition} {split}: {item['verified']}/{item['expected']}")

print()
print("discrepancias de trazabilidad (ids, particiones, rutas): 0")
print("discrepancias geométricas (bounding box, margen, componentes): 0")
print("discrepancias de dimensiones (esperadas / manifiesto / PNG): 0")
print("discrepancias de contenido del recorte: 0")
print("(cualquier discrepancia habría detenido la verificación indicando el image_id)")

print()
for split in SPLITS:
    ids = sorted(fallback_ids[("B", split)])
    detail = f" ({', '.join(str(i) for i in ids)})" if ids else ""
    print(f"fallbacks B {split}: {len(ids)}{detail}")

print(
    "máscaras B con más de una componente (keep_largest actuó) -> "
    + ", ".join(f"{split}: {stats[('B', split)]['multicomponent']}" for split in SPLITS)
)

print()
print("máscaras vacías C: 0")
print("fallbacks C: 0")
print(
    "máscaras de referencia C con más de una componente (ROI cubre todas) -> "
    + ", ".join(f"{split}: {stats[('C', split)]['multicomponent']}" for split in SPLITS)
)

print()
for condition in CONDITIONS:
    n_test = len(list((ROI_DIRS[condition] / "test" / "images").glob("*"))) if (ROI_DIRS[condition] / "test" / "images").exists() else 0
    print(f"ROI de test {condition} (no utilizadas en R5): {n_test}")

Verificación de ROI (solo lectura)
------------------------------------------------------------
muestras verificadas B train: 800/800
muestras verificadas B validation: 200/200
muestras verificadas C train: 800/800
muestras verificadas C validation: 200/200

discrepancias de trazabilidad (ids, particiones, rutas): 0
discrepancias geométricas (bounding box, margen, componentes): 0
discrepancias de dimensiones (esperadas / manifiesto / PNG): 0
discrepancias de contenido del recorte: 0
(cualquier discrepancia habría detenido la verificación indicando el image_id)

fallbacks B train: 0
fallbacks B validation: 2 (775, 869)
máscaras B con más de una componente (keep_largest actuó) -> train: 29, validation: 94

máscaras vacías C: 0
fallbacks C: 0
máscaras de referencia C con más de una componente (ROI cubre todas) -> train: 9, validation: 3

ROI de test B (no utilizadas en R5): 0
ROI de test C (no utilizadas en R5): 0


## Limitaciones que deben considerarse al interpretar B y C

- **Condición B.** Las ROI de B de `train` fueron generadas por un segmentador que se entrenó con esas mismas 800 imágenes, mientras que las ROI de B de `validation` corresponden a inferencia sobre imágenes no utilizadas durante su ajuste. Esto puede producir una diferencia en la calidad de las ROI entre `train` y `validation` (el Dice del segmentador es de 0.951 en `train` y 0.715 en `validation`; los dos casos con fallback aparecen únicamente en `validation`). Esa diferencia debe considerarse al interpretar los resultados de B. No se modifica el procedimiento ni se generan predicciones out-of-fold.
- **Condición C.** Es una condición idealizada basada en máscaras de referencia; sirve como referencia experimental y no constituye una alternativa operativa.
- **Escala efectiva.** Las ROI tienen un tamaño menor que la imagen completa y parte de ellas se amplía al llevarlas a 224 × 224, mientras que la imagen completa siempre se reduce. Es una propiedad inherente a la representación de entrada, no un artefacto del procesamiento.